In [1]:
import pandas as pd
import os
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

root_path = os.path.dirname((os.getcwd()))

In [ ]:
data_path = os.path.join(Path(os.getcwd()).resolve().parents[1], "dataset.xlsx")
df = pd.read_excel(data_path)

In [3]:
df.head(3)

,studentID,classID,timeStamp,studentEmotion,finalScore,totalImages,learningTimes,finishedLession,avgTimeLearn,avgTimeFinish,percentageFinish,timeInWeek,inTime,outTime,inWeekday,outWeekday
0,student_001,GENE1001-2-3-24(N01),2025-08-18 04:11:05.921000,neutral,10.0,263,214,87,12.7,31.2,49.1,2714.7,2033.4,681.3,2589.2,125.5
1,student_001,GENE1001-2-3-24(N01),2025-08-18 04:11:43.762000,neutral,10.0,263,214,87,12.7,31.2,49.1,2714.7,2033.4,681.3,2589.2,125.5
2,student_001,GENE1001-2-3-24(N01),2025-08-18 04:11:43.808000,neutral,10.0,263,214,87,12.7,31.2,49.1,2714.7,2033.4,681.3,2589.2,125.5


In [4]:
df.describe()

,finalScore,totalImages,learningTimes,finishedLession,avgTimeLearn,avgTimeFinish,percentageFinish,timeInWeek,inTime,outTime,inWeekday,outWeekday
count,12095.000000,12095.000000,12095.000000,12095.000000,12095.000000,12095.000000,12095.000000,12095.000000,12095.000000,12095.000000,12095.000000,12095.000000
mean,8.140814,175.323522,99.201323,44.500124,10.050508,19.171277,60.005539,924.517983,562.206507,362.311476,681.091848,243.426135
std,2.088232,117.693367,63.929325,21.251072,16.710562,29.906770,16.612644,1408.891266,894.763634,940.605427,1348.985372,397.342973
min,0.000000,3.000000,1.000000,0.000000,0.200000,0.000000,0.000000,0.200000,0.200000,0.000000,0.000000,0.000000
25%,7.600000,82.000000,45.000000,26.000000,4.500000,10.900000,48.100000,308.100000,180.300000,96.000000,204.200000,29.500000
50%,8.400000,140.000000,75.000000,36.000000,7.100000,13.600000,59.900000,475.400000,283.800000,169.800000,317.000000,125.500000
75%,9.250000,251.000000,155.000000,66.000000,9.400000,17.100000,72.800000,865.100000,550.600000,311.700000,551.000000,302.900000
max,10.000000,524.000000,257.000000,87.000000,155.700000,346.800000,100.000000,10628.200000,8539.900000,9972.500000,10628.200000,2630.700000


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12095 entries, 0 to 12094
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   studentID         12095 non-null  object 
 1   classID           12095 non-null  object 
 2   timeStamp         12095 non-null  object 
 3   studentEmotion    12095 non-null  object 
 4   finalScore        12095 non-null  float64
 5   totalImages       12095 non-null  int64  
 6   learningTimes     12095 non-null  int64  
 7   finishedLession   12095 non-null  int64  
 8   avgTimeLearn      12095 non-null  float64
 9   avgTimeFinish     12095 non-null  float64
 10  percentageFinish  12095 non-null  float64
 11  timeInWeek        12095 non-null  float64
 12  inTime            12095 non-null  float64
 13  outTime           12095 non-null  float64
 14  inWeekday         12095 non-null  float64
 15  outWeekday        12095 non-null  float64
dtypes: float64(9), int64(3), object(4)
memor

In [6]:
df = df.drop_duplicates(subset=["studentID"])

In [7]:
def encode_emotion(df, emotion_col, id_col="studentID"):
    encoding_map = {
        "neutral": 0,
        "happy": 1,
        "sad": -1
    }

    df = df.copy()
    df[emotion_col + "_encoded"] = df[emotion_col].map(encoding_map)

    result = (
        df.groupby(id_col)[emotion_col + "_encoded"]
          .sum()
          .reset_index(name="emotion_sum")
    )

    return result

In [8]:
def log_transform(df, columns, eps=1e-6):
    df = df.copy()
    
    for col in columns:
        df[col] = np.log1p(np.clip(df[col], -1 + eps, None))
    
    return df

In [9]:
def z_score_transform(df, id_col):
    df = df.copy()
    cols_to_scale = df.columns.difference([id_col])
    scaler = StandardScaler()
    df.loc[:, cols_to_scale] = scaler.fit_transform(df[cols_to_scale])
    return df

def scale_transform(df, column):
    df = df.copy()
    scaler = MinMaxScaler(feature_range=(0, 10))
    df[column + "_scaled"] = scaler.fit_transform(df[[column]])
    df = df.drop([column], axis=1)
    return df

In [10]:
master_df = encode_emotion(df, emotion_col='studentEmotion', id_col='studentID')
master_df = master_df.merge(df, how="inner", on="studentID")
master_df = master_df.drop(["classID", "timeStamp","studentEmotion", "timeInWeek"], axis=1)

In [11]:
print(master_df.shape)
master_df.head(3)

(169, 13)


,studentID,emotion_sum,finalScore,totalImages,learningTimes,finishedLession,avgTimeLearn,avgTimeFinish,percentageFinish,inTime,outTime,inWeekday,outWeekday
0,student_001,0,10.0,263,214,87,12.7,31.2,49.1,2033.4,681.3,2589.2,125.5
1,student_002,0,7.6,259,161,67,9.0,21.6,49.6,1138.8,311.7,1229.4,221.1
2,student_003,-1,8.4,395,217,82,8.0,21.2,43.2,702.4,1038.2,1404.2,336.4


In [12]:
master_df.describe()

,emotion_sum,finalScore,totalImages,learningTimes,finishedLession,avgTimeLearn,avgTimeFinish,percentageFinish,inTime,outTime,inWeekday,outWeekday
count,169.000000,169.000000,169.000000,169.000000,169.000000,169.000000,169.000000,169.000000,169.000000,169.000000,169.000000,169.000000
mean,-0.295858,8.295858,121.538462,72.307692,36.686391,11.679290,20.433136,66.854438,451.417751,378.068639,658.799408,170.686982
std,0.552091,2.264605,100.922884,54.395269,19.207882,18.731155,35.646655,17.963925,870.887167,1103.360594,1490.812950,291.358533
min,-1.000000,0.000000,3.000000,1.000000,0.000000,0.200000,0.000000,0.000000,0.200000,0.000000,0.000000,0.000000
25%,-1.000000,8.000000,52.000000,37.000000,22.000000,5.400000,11.200000,53.900000,160.900000,83.200000,201.600000,8.600000
50%,0.000000,9.000000,94.000000,53.000000,30.000000,8.100000,13.600000,67.400000,240.200000,154.100000,283.900000,89.800000
75%,0.000000,9.500000,154.000000,84.000000,44.000000,12.100000,16.600000,79.800000,392.500000,266.700000,445.900000,195.200000
max,1.000000,10.000000,524.000000,257.000000,87.000000,155.700000,346.800000,100.000000,8539.900000,9972.500000,10628.200000,2630.700000


In [13]:
log_cols = master_df.drop(["studentID", "finalScore"], axis=1).columns

In [14]:
master_df.head(5)

,studentID,emotion_sum,finalScore,totalImages,learningTimes,finishedLession,avgTimeLearn,avgTimeFinish,percentageFinish,inTime,outTime,inWeekday,outWeekday
0,student_001,0,10.0,263,214,87,12.7,31.2,49.1,2033.4,681.3,2589.2,125.5
1,student_002,0,7.6,259,161,67,9.0,21.6,49.6,1138.8,311.7,1229.4,221.1
2,student_003,-1,8.4,395,217,82,8.0,21.2,43.2,702.4,1038.2,1404.2,336.4
3,student_004,0,8.0,203,131,61,21.4,46.0,53.1,455.2,2349.2,1827.3,977.1
4,student_005,-1,6.0,280,136,64,8.1,17.3,51.5,275.6,828.7,959.1,145.2


In [15]:
log_cols

Index(['emotion_sum', 'totalImages', 'learningTimes', 'finishedLession',
       'avgTimeLearn', 'avgTimeFinish', 'percentageFinish', 'inTime',
       'outTime', 'inWeekday', 'outWeekday'],
      dtype='object')

In [16]:
log_transform(df=master_df, columns=log_cols)

,studentID,emotion_sum,finalScore,totalImages,learningTimes,finishedLession,avgTimeLearn,avgTimeFinish,percentageFinish,inTime,outTime,inWeekday,outWeekday
0,student_001,0.000000,10.00,5.575949,5.370638,4.477337,2.617396,3.471966,3.914021,7.617956,6.525469,7.859490,4.840242
1,student_002,0.000000,7.60,5.560682,5.087596,4.219508,2.302585,3.117950,3.923952,7.038608,5.745244,7.115095,5.403128
2,student_003,-13.815511,8.40,5.981414,5.384495,4.418841,2.197225,3.100092,3.788725,6.555926,6.946206,7.247935,5.821269
3,student_004,0.000000,8.00,5.318120,4.882802,4.127134,3.109061,3.850148,3.990834,6.122931,7.762256,7.511142,6.885612
4,student_005,-13.815511,6.00,5.638355,4.919981,4.174387,2.208274,2.906901,3.960813,5.622572,6.721064,6.867037,4.984976
...,...,...,...,...,...,...,...,...,...,...,...,...,...
164,student_165,0.000000,9.75,3.737670,3.332205,3.044522,3.010621,3.295837,4.345103,5.975081,4.856707,5.804834,5.248076
165,student_166,0.000000,0.00,1.386294,0.693147,0.000000,0.182322,0.000000,0.000000,0.182322,0.000000,0.000000,0.182322
166,student_167,-13.815511,8.25,2.890372,3.988984,3.091042,2.041220,2.884801,3.919991,5.619676,4.394449,5.416100,4.881286
167,student_168,0.000000,9.50,5.176150,4.564348,3.258097,1.504077,2.646175,3.597312,5.128715,5.083886,3.813307,5.652138


In [18]:
df_transformed = log_transform(df=master_df, columns=log_cols)
df_transformed = z_score_transform(df=df_transformed, id_col="studentID")

In [19]:
print(df_transformed.shape)
df_transformed.head(3)

(169, 13)


,studentID,emotion_sum,finalScore,totalImages,learningTimes,finishedLession,avgTimeLearn,avgTimeFinish,percentageFinish,inTime,outTime,inWeekday,outWeekday
0,student_001,0.715094,0.754748,1.293228,1.905885,1.819326,0.587077,1.155823,-0.579566,1.855016,1.125628,1.979479,0.497655
1,student_002,0.715094,-0.308189,1.274963,1.493427,1.338519,0.096610,0.590586,-0.556346,1.357795,0.574302,1.279082,0.750887
2,student_003,-1.383062,0.046123,1.778293,1.926077,1.710241,-0.067539,0.562073,-0.872532,0.943537,1.422931,1.404071,0.939002


In [20]:
df_transformed.describe()

,emotion_sum,finalScore,totalImages,learningTimes,finishedLession,avgTimeLearn,avgTimeFinish,percentageFinish,inTime,outTime,inWeekday,outWeekday
count,1.690000e+02,1.690000e+02,1.690000e+02,1.690000e+02,1.690000e+02,1.690000e+02,1.690000e+02,1.690000e+02,1.690000e+02,1.690000e+02,1.690000e+02,1.690000e+02
mean,-6.832142e-17,-1.051099e-16,7.147471e-16,-8.408790e-17,-7.988350e-16,2.732857e-16,-5.255494e-16,-1.828912e-15,-7.567911e-16,2.732857e-16,1.681758e-16,-2.312417e-16
std,1.002972e+00,1.002972e+00,1.002972e+00,1.002972e+00,1.002972e+00,1.002972e+00,1.002972e+00,1.002972e+00,1.002972e+00,1.002972e+00,1.002972e+00,1.002972e+00
min,-1.383062e+00,-3.674156e+00,-3.718933e+00,-4.910302e+00,-6.530132e+00,-3.206703e+00,-4.387660e+00,-9.731275e+00,-4.526555e+00,-3.485426e+00,-5.415461e+00,-1.679883e+00
25%,-1.383062e+00,-1.310327e-01,-6.276492e-01,-6.195730e-01,-6.829783e-01,-5.986931e-01,-3.937658e-01,-3.656398e-01,-3.171743e-01,-3.528237e-01,-4.181579e-01,-6.623566e-01
50%,7.150944e-01,3.118577e-01,7.050421e-02,-1.075050e-01,-1.263405e-01,-5.032335e-02,-1.070321e-01,1.484325e-01,2.496164e-02,7.883539e-02,-9.740312e-02,3.484819e-01
75%,7.150944e-01,5.333030e-01,6.561597e-01,5.535931e-01,5.686344e-01,5.173051e-01,1.913439e-01,5.379840e-01,4.450319e-01,4.645086e-01,3.261838e-01,6.951050e-01
max,8.203624e-01,7.547482e-01,2.115636e+00,2.171569e+00,1.819326e+00,4.383759e+00,4.955287e+00,1.059735e+00,3.086305e+00,3.020947e+00,3.307898e+00,1.863112e+00


In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
target_col = "finalScore"
X = df_transformed.drop(columns=["studentID", "finalScore"], axis=1)
y = df_transformed[target_col]

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train = X_train.values
X_test = X_test.values

In [24]:
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Dense, Dropout, LayerNormalization,
    MultiHeadAttention, Reshape, Flatten
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

In [25]:
def transformer_encoder(x, head_size, num_heads, ff_dim, dropout=0.1):
    # Self-attention
    attn_output = MultiHeadAttention(
        key_dim=head_size,
        num_heads=num_heads,
        dropout=dropout
    )(x, x)
    x = LayerNormalization(epsilon=1e-6)(x + attn_output)

    # Feed-forward network
    ff = Dense(ff_dim, activation="relu")(x)
    ff = Dropout(dropout)(ff)
    ff = Dense(x.shape[-1])(ff)

    return LayerNormalization(epsilon=1e-6)(x + ff)


In [26]:
n_features = X_train.shape[1]

inputs = Input(shape=(n_features,))

# Convert features to tokens
x = Reshape((n_features, 1))(inputs)

# Linear embedding per feature
x = Dense(32)(x)   # embedding_dim = 32

# Transformer blocks
for _ in range(2):  # number of transformer layers
    x = transformer_encoder(
        x,
        head_size=32,
        num_heads=4,
        ff_dim=64,
        dropout=0.2
    )

# Flatten token representations
x = Flatten()(x)

# Regression head
x = Dense(32, activation="relu", kernel_regularizer=l2(1e-4))(x)
x = Dropout(0.3)(x)
x = Dense(16, activation="relu", kernel_regularizer=l2(1e-4))(x)
x = Dropout(0.2)(x)

output = Dense(1, activation="linear")(x)

model = Model(inputs=inputs, outputs=output)

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

model.summary()


Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 11)]         0           []                               
                                                                                                  
 reshape (Reshape)              (None, 11, 1)        0           ['input_1[0][0]']                
                                                                                                  
 dense (Dense)                  (None, 11, 32)       64          ['reshape[0][0]']                
                                                                                                  
 multi_head_attention (MultiHea  (None, 11, 32)      16800       ['dense[0][0]',                  
 dAttention)                                                      'dense[0][0]']              

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor="val_loss",        
    patience=10,               
    min_delta=1e-4,            
    restore_best_weights=True, 
    verbose=1
)

In [28]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/100
5/5 [==============================] - 2s 72ms/step - loss: 2.3124 - mae: 1.2074 - val_loss: 1.2617 - val_mae: 0.7582
Epoch 2/100
5/5 [==============================] - 0s 13ms/step - loss: 1.3556 - mae: 0.8015 - val_loss: 1.0566 - val_mae: 0.6204
Epoch 3/100
5/5 [==============================] - 0s 12ms/step - loss: 1.3374 - mae: 0.7175 - val_loss: 1.0122 - val_mae: 0.5852
Epoch 4/100
5/5 [==============================] - 0s 12ms/step - loss: 1.1623 - mae: 0.6570 - val_loss: 1.0048 - val_mae: 0.6069
Epoch 5/100
5/5 [==============================] - 0s 13ms/step - loss: 0.9522 - mae: 0.5834 - val_loss: 0.9951 - val_mae: 0.6155
Epoch 6/100
5/5 [==============================] - 0s 14ms/step - loss: 1.1364 - mae: 0.6629 - val_loss: 0.9771 - val_mae: 0.5957
Epoch 7/100
5/5 [==============================] - 0s 13ms/step - loss: 1.0091 - mae: 0.6210 - val_loss: 0.9727 - val_mae: 0.5937
Epoch 8/100
5/5 [==============================] - 0s 12ms/step - loss: 1.0188 - mae: 0.59

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

In [30]:
# Predict
y_pred = model.predict(X_test).flatten()

# Metrics
mse  = mean_squared_error(y_test, y_pred)
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print("Evaluation Results (Transformer Regression)")
print(f"MSE  : {mse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R²   : {r2:.4f}")

2/2 [==============================] - 0s 2ms/step
Evaluation Results (Transformer Regression)
MSE  : 0.7337
MAE  : 0.5013
R²   : 0.2716
